In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window

B = "workspace.olist_bronze"
S = "workspace.olist_silver_py"

def dedupe(df, keys, order_col="_ingested_at", desc=False):
    ordering = F.col(order_col).desc() if desc else F.col(order_col).asc()
    w = Window.partitionBy(*keys).orderBy(ordering)
    return df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

def save(df,name):
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{S}.{name}")
        print(name, spark.table(f"{S}.{name}").count())


In [0]:
df = dedupe(spark.table(f"{B}.bronze_customers"), ["customer_id"]).select(
    "customer_id", "customer_unique_id",
    F.col("customer_zip_code_prefix").alias("zip_prefix"),
    F.lower(F.trim("customer_city")).alias("city"),
    F.upper(F.trim("customer_state")).alias("state"),
    "_ingested_at")
save(df, "customers")

customers 99441


In [0]:
df = dedupe(spark.table(f"{B}.bronze_sellers"), ["seller_id"]).select(
    "seller_id",
    F.col("seller_zip_code_prefix").alias("zip_prefix"),
    F.lower(F.trim("seller_city")).alias("city"),
    F.upper(F.trim("seller_state")).alias("state"),
    "_ingested_at")
save(df, "sellers")

sellers 3095


In [0]:
df = (spark.table(f"{B}.bronze_product_category_name_translation")
      .select(F.lower(F.trim("product_category_name")).alias("category_name_pt"),
              F.lower(F.trim("product_category_name_english")).alias("category_name_en"))
      .distinct())
save(df, "category_translation")

category_translation 71


In [0]:
cat = F.lower(F.trim("product_category_name"))
df = dedupe(spark.table(f"{B}.bronze_products"), ["product_id"]).select(
    "product_id",
    F.when(cat == "", None).otherwise(cat).alias("category_name_pt"),
    F.expr("try_cast(product_photos_qty AS INT)").alias("photos_qty"),
    F.expr("try_cast(product_weight_g AS INT)").alias("weight_g"),
    F.expr("try_cast(product_length_cm AS INT)").alias("length_cm"),
    F.expr("try_cast(product_height_cm AS INT)").alias("height_cm"),
    F.expr("try_cast(product_width_cm AS INT)").alias("width_cm"),
    "_ingested_at")
save(df, "products")


products 32951


In [0]:
df = dedupe(spark.table(f"{B}.bronze_orders"), ["order_id"]).select(
    "order_id", "customer_id",
    F.lower(F.trim("order_status")).alias("order_status"),
    F.expr("try_cast(order_purchase_timestamp AS TIMESTAMP)").alias("purchased_at"),
    F.expr("try_cast(order_approved_at AS TIMESTAMP)").alias("approved_at"),
    F.expr("try_cast(order_delivered_carrier_date AS TIMESTAMP)").alias("shipped_at"),
    F.expr("try_cast(order_delivered_customer_date AS TIMESTAMP)").alias("delivered_at"),
    F.expr("try_cast(order_estimated_delivery_date AS TIMESTAMP)").alias("estimated_delivery_at"),
    "_ingested_at")
save(df, "orders")

orders 99441


In [0]:
df = dedupe(spark.table(f"{B}.bronze_order_items"), ["order_id", "order_item_id"]).select(
    "order_id",
    F.expr("try_cast(order_item_id AS INT)").alias("order_item_id"),
    "product_id", "seller_id",
    F.expr("try_cast(shipping_limit_date AS TIMESTAMP)").alias("shipping_limit_at"),
    F.expr("try_cast(price AS DECIMAL(10,2))").alias("price"),
    F.expr("try_cast(freight_value AS DECIMAL(10,2))").alias("freight_value"),
    "_ingested_at")
save(df, "order_items")

order_items 112650


In [0]:
df = dedupe(spark.table(f"{B}.bronze_order_payments"), ["order_id", "payment_sequential"]).select(
    "order_id",
    F.expr("try_cast(payment_sequential As INT)").alias("payment_sequential"),
    F.lower(F.trim("payment_type")).alias("payment_type"),
    F.expr("try_cast(payment_installments AS INT)").alias("installments"),
    F.expr("try_cast(payment_value AS DECIMAL(10,2))").alias("payment_value"),
    "_ingested_at")
save(df, "payments")


payments 103886


In [0]:
base = spark.table(f"{B}.bronze_order_reviews").select(
    "review_id", "order_id",
    F.expr("try_cast(review_score AS INT)").alias("review_score"),
    F.expr("try_cast(review_creation_date AS TIMESTAMP)").alias("created_at"),
    F.expr("try_cast(review_answer_timestamp AS TIMESTAMP)").alias("answered_at"),
    "_ingested_at")
df = dedupe(base, ["review_id", "order_id"], order_col="answered_at", desc=True)
save(df, "reviews")

reviews 99224


In [0]:
tables = ["customers", "sellers", "products", "orders",
          "order_items", "payments", "reviews", "category_translation"]
for t in tables:
    a = spark.table(f"workspace.olist_silver_sql.{t}").count()
    b = spark.table(f"workspace.olist_silver_py.{t}").count()
    print(f"{t:22} sql={a:>7} py={b:>7} {'OK' if a==b else 'MISMATCH'}")

customers              sql=  99441 py=  99441 OK
sellers                sql=   3095 py=   3095 OK
products               sql=  32951 py=  32951 OK
orders                 sql=  99441 py=  99441 OK
order_items            sql= 112650 py= 112650 OK
payments               sql= 103886 py= 103886 OK
reviews                sql=  99224 py=  99224 OK
category_translation   sql=     71 py=     71 OK


In [0]:
%sql
SELECT 'sql' AS track,
    (SELECT COUNT(purchased_at) FROM workspace.olist_silver_sql.orders) AS orders_with_date,
    (SELECT COUNT(price) FROM workspace.olist_silver_sql.order_items) AS items_with_price
UNION ALL
SELECT 'py',
    (SELECT COUNT(purchased_at) FROM workspace.olist_silver_py.orders),
    (SELECT COUNT(price) FROM workspace.olist_silver_py.order_items);

track,orders_with_date,items_with_price
sql,99441,112650
py,99441,112650


In [0]:
%sql
DESCRIBE workspace.olist_silver_sql.order_items;

col_name,data_type,comment
order_id,string,null
order_item_id,int,null
product_id,string,null
seller_id,string,null
shipping_limit_at,timestamp,null
freight_value,"decimal(10,2)",null
_ingested_at,timestamp,null
